In [1]:
import netCDF4 as nc
import numpy as np
import torch

ImportError: /home/aidl/anaconda3/envs/notebook-env/lib/python3.10/site-packages/torch/lib/libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent

In [2]:
WAVE_VARS = [
    'VPED', 'VSDX', 'VSDY', 'VTM01_SW1', 'VTM01_SW2', 'VTM01_WW', 'VTM02', 'VTM10', 'VTPK'   # ← total = 9
]

WAVE_VARS1 = [
    'VMDR_SW1', 'VCMX', 'VHM0', 'VHM0_SW1', 'VHM0_SW2', 'VHM0_WW', 'VMDR', 'VMDR_SW2', 'VMDR_WW', 'VMXL'   # ← total = 10
]

EXTRA_VARS = ["TLA", "TUAX", "TUAY"]  # +3

In [3]:
ds = nc.Dataset(r'new_dAtA/after_xl.nc')
ds1 = nc.Dataset(r'new_dAtA/till xl.nc')
ds3 = nc.Dataset(r'new_dAtA/tla_taux_tauy.nc')

time_dim = ds.dimensions["time"].size
lat_dim = ds.dimensions["latitude"].size
lon_dim = ds.dimensions["longitude"].size

def load_time_slice(ds, var_list, t):
    data = []
    for v in var_list:
        data.append(ds.variables[v][t, :, :])
    return np.stack(data, axis=0)  # (C, H, W)

In [7]:
bathy_ds = nc.Dataset("new_dAtA/cmems_GEBCO_resampled_new.nc")

bathymetry = np.stack([
    bathy_ds.variables["latitude"][:],
    bathy_ds.variables["longitude"][:],
    bathy_ds.variables["deptho"][:]
], axis=0)  # (3, H, W)

ValueError: all input arrays must have the same shape

In [ ]:
def build_input_tensor(t):
    wave_9 = load_time_slice(ds, WAVE_VARS, t)        # (9, H, W)
    wave_10 = load_time_slice(ds1, WAVE_VARS1, t)        # (10, H, W)
    extra_3 = load_time_slice(ds3, EXTRA_VARS, t)       # (3, H, W)

    x = np.concatenate(
        [wave_9, wave_10, extra_3, bathymetry],
        axis=0
    )  # (25, H, W)

    return torch.tensor(x, dtype=torch.float32)

In [ ]:
import torch.nn as nn
import segmentation_models_pytorch as smp

In [ ]:
class DeepLabEmbedding(nn.Module):
    def __init__(self, in_channels=25, embedding_dim=128):
        super().__init__()

        # Channel bottleneck (VERY IMPORTANT)
        self.channel_mixer = nn.Conv2d(
            in_channels, 16, kernel_size=1, bias=False
        )

        self.backbone = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=16,
            classes=1
        )

        self.encoder = self.backbone.encoder
        self.aspp = self.backbone.decoder.aspp

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.embed = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.channel_mixer(x)
        x = self.encoder(x)[-1]
        x = self.aspp(x)
        x = self.pool(x)
        return self.embed(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepLabEmbedding(
    in_channels=25,
    embedding_dim=128
).to(device)

model.eval()

In [ ]:
embeddings = []

with torch.no_grad():
    for t in range(time_dim):
        x_t = build_input_tensor(t)
        x_t = x_t.unsqueeze(0).to(device)  # (1, 25, H, W)

        z_t = model(x_t)                   # (1, 128)
        embeddings.append(z_t.cpu())

In [ ]:
Z = torch.cat(embeddings, dim=0)  # (T, 128)

In [ ]:
np.save("deeplab_embeddings.npy", Z.numpy())